In [1]:
import re
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from IPython.display import display
from nltk.tokenize import sent_tokenize
from scipy.sparse import coo_matrix, csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec, FastText
from time import perf_counter

nltk.data.find('tokenizers/punkt_tab/spanish/')

FileSystemPathPointer('C:\\Users\\valde\\AppData\\Roaming\\nltk_data\\tokenizers\\punkt_tab\\spanish')

---

### 1. Preparación de los corpus

In [2]:
data = Path('data')
ruta_noticias = data / 'df_total.csv'
ruta_quijote = data / 'don-quijote.txt'
ruta_suc = data / 'embeddings-m-model.vec'

noticias = pd.read_csv(ruta_noticias, encoding='utf-8-sig')
columnas = {str(c).lower().strip(): c for c in noticias.columns}
columna_texto = next((columnas[c] for c in
                      ['news', 'texto', 'text', 'noticia', 'contenido', 'content',
                       'body', 'article', 'titular', 'title'] if c in columnas), None)

documentos_noticias = noticias[columna_texto].dropna().astype(str).str.strip()
documentos_noticias = documentos_noticias[documentos_noticias != ''].tolist()

texto_original_quijote = ruta_quijote.read_text(encoding='utf-8-sig')
titulo_quijote = 'El ingenioso hidalgo don Quijote de la Mancha'
cierre_quijote = "End of Project Gutenberg's Don Quijote, by Miguel de Cervantes Saavedra"

patron_inicio = re.compile(r'El\s+ingenioso\s+hidalgo\s+don\s+Quijote\s+de\s+la\s+Mancha', re.I)
patron_final = re.compile(
    r"End\s+of\s+Project\s+Gutenberg['’]s\s+Don\s+Quijote,\s+by\s+Miguel\s+de\s+Cervantes\s+Saavedra",
    re.I
)
coincidencia_inicio = patron_inicio.search(texto_original_quijote)
coincidencias_final = list(patron_final.finditer(texto_original_quijote, coincidencia_inicio.end()))
coincidencia_final = coincidencias_final[-1]

cuerpo_quijote = texto_original_quijote[coincidencia_inicio.end():coincidencia_final.start()].strip()
texto_quijote = titulo_quijote + '\n' + cuerpo_quijote + '\n' + cierre_quijote
documentos_quijote = [texto_quijote]

assert texto_quijote.startswith(titulo_quijote)
assert texto_quijote.endswith(cierre_quijote)

print(f'Documentos de noticias: {len(documentos_noticias):,}')

Documentos de noticias: 1,213


In [3]:
patron_urls = re.compile(r'(?:https?://|www\.)\S+', flags=re.IGNORECASE)
patron_tokens = re.compile(r'[^\W_]+', flags=re.UNICODE)

def tokenizar(texto):
    texto = patron_urls.sub(' ', texto.lower())
    return patron_tokens.findall(texto)

def preparar_corpus(documentos):
    oraciones = []
    for documento in documentos:
        partes = sent_tokenize(documento, language='spanish')
        for parte in partes:
            tokens = tokenizar(parte)
            if tokens:
                oraciones.append(tokens)
    return oraciones

oraciones_noticias = preparar_corpus(documentos_noticias)

oraciones_quijote = (
    [tokenizar(titulo_quijote)]
    + preparar_corpus([cuerpo_quijote])
    + [tokenizar(cierre_quijote)]
)
assert oraciones_quijote[0] == tokenizar(titulo_quijote)
assert oraciones_quijote[-1] == tokenizar(cierre_quijote)

**¿Qué normalización se utilizó?**

Se convirtieron las palabras a minúsculas y se eliminaron URLs y puntuación, pero se conservaron los números y las tildes. Esto permite reducir variantes innecesarias sin perder información que puede ser importante en noticias, como cifras o diferencias entre palabras. Además, en el vocabulario SUC cargado de 200,000 términos aparecen “españa”, “espana”, “año” y “ano”, pero no “España”, por lo que usar minúsculas y mantener tildes resulta compatible con esta comparación.

**¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?**

Para este laboratorio no conviene. Aunque en el Lab 1 se eliminaron stopwords y se lematizó para reducir el vocabulario, aquí las palabras vecinas son las que permiten aprender el contexto. Si quitamos palabras como “de” o “la”, las ventanas juntarían palabras que originalmente no estaban cerca; además, la lematización podría hacer que formas distintas, como “gobiernos” y “gobernó”, pierdan parte de su información gramatical.

In [4]:
frecuencias_noticias = Counter(t for oracion in oraciones_noticias for t in oracion)
frecuencias_quijote = Counter(t for oracion in oraciones_quijote for t in oracion)

total_tokens_noticias = sum(frecuencias_noticias.values())
total_tokens_quijote = sum(frecuencias_quijote.values())

tabla_corpus = pd.DataFrame([
    ['Noticias', len(documentos_noticias), len(oraciones_noticias),
     total_tokens_noticias, len(frecuencias_noticias)],
    ['Don Quijote', len(documentos_quijote), len(oraciones_quijote),
     total_tokens_quijote, len(frecuencias_quijote)]
], columns=['Corpus', 'Documentos', 'Oraciones', 'Tokens', 'Tipos'])

display(tabla_corpus)

,Corpus,Documentos,Oraciones,Tokens,Tipos
0,Noticias,1213,15423,637537,29518
1,Don Quijote,1,9514,381238,22948


In [5]:
def vocabulario_min_count(nombre, frecuencias):
    filas = []
    total = sum(frecuencias.values())
    for min_count in [1, 2, 5]:
        vocabulario = {palabra for palabra, f in frecuencias.items() if f >= min_count}
        tokens_fuera = sum(f for f in frecuencias.values() if f < min_count)
        porcentaje = 100 * tokens_fuera / total if total else 0
        filas.append([nombre, min_count, len(vocabulario), tokens_fuera, round(porcentaje, 2)])
    return filas


resultados_vocabulario = (
    vocabulario_min_count('Noticias', frecuencias_noticias)
    + vocabulario_min_count('Don Quijote', frecuencias_quijote)
)
tabla_vocabulario = pd.DataFrame(
    resultados_vocabulario,
    columns=['Corpus', 'min_count', 'Tamaño vocabulario', 'Tokens fuera', '% tokens fuera']
)
display(tabla_vocabulario)

,Corpus,min_count,Tamaño vocabulario,Tokens fuera,% tokens fuera
0,Noticias,1,29518,0,0.00
1,Noticias,2,17135,12383,1.94
2,Noticias,5,8888,34346,5.39
3,Don Quijote,1,22948,0,0.00
4,Don Quijote,2,11760,11188,2.93
5,Don Quijote,5,5301,28126,7.38


In [6]:
limite_suc = 200_000

if 'suc' not in globals():
    suc = KeyedVectors.load_word2vec_format(str(ruta_suc), limit=limite_suc)

print(f'Vectores SUC cargados: {len(suc.key_to_index):,}')
print(f'Dimensiones: {suc.vector_size}')

palabras = ['españa', 'España', 'espana', 'año', 'ano']
tabla_suc = pd.DataFrame([
    [palabra, palabra in suc.key_to_index] for palabra in palabras
], columns=['Palabra', '¿Existe en SUC cargado?'])
display(tabla_suc)


Vectores SUC cargados: 200,000
Dimensiones: 100


,Palabra,¿Existe en SUC cargado?
0,españa,True
1,España,False
2,espana,True
3,año,True
4,ano,True


In [7]:
def pares_skipgram(tokens, ventana=2):
    pares = []
    for i, palabra in enumerate(tokens):
        inicio = max(0, i - ventana)
        fin = min(len(tokens), i + ventana + 1)
        for j in range(inicio, fin):
            if i != j:
                pares.append((palabra, tokens[j]))
    return pares

oracion_ejemplo = next((s for s in oraciones_noticias if 6 <= len(s) <= 12), None)

pares_ejemplo = pares_skipgram(oracion_ejemplo, ventana=2)
print('Oración:', ' '.join(oracion_ejemplo))
print(f'Tokens: {len(oracion_ejemplo)} | Pares: {len(pares_ejemplo)}')
display(pd.DataFrame(pares_ejemplo, columns=['Palabra central', 'Contexto']))

Oración: como último punto se debe garantizar que los avances lleguen a todos
Tokens: 12 | Pares: 42


,Palabra central,Contexto
0,como,último
1,como,punto
2,último,como
3,último,punto
4,último,se
5,punto,como
6,punto,último
7,punto,se
8,punto,debe
9,se,último


In [8]:
def contar_pares(oraciones, ventana):
    total = 0
    for oracion in oraciones:
        n = len(oracion)
        for distancia in range(1, min(ventana, n - 1) + 1):
            total += 2 * (n - distancia)
    return total

assert contar_pares([oracion_ejemplo], 2) == len(pares_ejemplo)

resultados_pares = []
for ventana in [2, 5]:
    total_pares = contar_pares(oraciones_noticias, ventana)
    razon = total_pares / total_tokens_noticias if total_tokens_noticias else 0
    resultados_pares.append([ventana, total_tokens_noticias, total_pares, round(razon, 2)])

tabla_pares = pd.DataFrame(
    resultados_pares,
    columns=['Ventana', 'Tokens del corpus', 'Pares skip-gram', 'Pares por token']
)
display(tabla_pares)

,Ventana,Tokens del corpus,Pares skip-gram,Pares por token
0,2,637537,2457706,3.86
1,5,637537,5914404,9.28


**¿Por qué Word2Vec no necesita datos etiquetados?**

Porque aprende de las propias palabras del texto: toma una palabra central y utiliza las palabras cercanas como ejemplos de contexto. Con las 637,537 palabras del corpus de noticias se generaron 2,457,706 pares con ventana 2 y 5,914,404 con ventana 5. Estos pares se obtienen automáticamente, sin que alguien tenga que asignar una categoría o etiqueta a cada noticia.

---

### 2. Co-ocurrencia, PPMI y SVD

In [9]:
vocabulario_2000 = [palabra for palabra, _ in frecuencias_noticias.most_common(2000)]
indice_palabras = {palabra: i for i, palabra in enumerate(vocabulario_2000)}

def construir_coocurrencias(oraciones, indices, ventana=2):
    n = len(indices)
    matriz = csr_matrix((n, n), dtype=np.int32)
    filas, columnas = [], []

    for oracion in oraciones:
        ids = [indices.get(palabra, -1) for palabra in oracion]
        for distancia in range(1, ventana + 1):
            for i in range(len(ids) - distancia):
                a, b = ids[i], ids[i + distancia]
                if a == -1 or b == -1:
                    continue
                filas.extend([a, b])
                columnas.extend([b, a])

                if len(filas) >= 200_000:
                    bloque = coo_matrix(
                        (np.ones(len(filas), dtype=np.int32), (filas, columnas)),
                        shape=(n, n)
                    ).tocsr()
                    matriz += bloque
                    filas.clear()
                    columnas.clear()

    if filas:
        matriz += coo_matrix(
            (np.ones(len(filas), dtype=np.int32), (filas, columnas)),
            shape=(n, n)
        ).tocsr()

    return matriz


matriz_cooc = construir_coocurrencias(oraciones_noticias, indice_palabras, ventana=2)
porcentaje_ceros_cooc = 100 * (1 - matriz_cooc.nnz / (matriz_cooc.shape[0] * matriz_cooc.shape[1]))

print('Forma de la matriz:', matriz_cooc.shape)
print(f'Ceros: {porcentaje_ceros_cooc:.2f}%')
print(f'Entradas distintas de cero: {matriz_cooc.nnz:,}')

Forma de la matriz: (2000, 2000)
Ceros: 93.51%
Entradas distintas de cero: 259,489


In [10]:
vectorizador = CountVectorizer(
    vocabulary=indice_palabras,
    tokenizer=tokenizar,
    token_pattern=None,
    lowercase=False
)
matriz_bow = vectorizador.transform(documentos_noticias)

comparacion = pd.DataFrame([
    ['Co-ocurrencia', str(matriz_cooc.shape), porcentaje_ceros_cooc],
    ['BoW (noticias)', str(matriz_bow.shape),
     100 * (1 - matriz_bow.nnz / (matriz_bow.shape[0] * matriz_bow.shape[1]))]
], columns=['Matriz', 'Forma', '% de ceros'])
display(comparacion.round(2))
mas_dispersa = comparacion.loc[comparacion['% de ceros'].idxmax(), 'Matriz']
print(f'En esta comparación, la matriz más dispersa es: {mas_dispersa}.')

,Matriz,Forma,% de ceros
0,Co-ocurrencia,"(2000, 2000)",93.51
1,BoW (noticias),"(1213, 2000)",91.88


En esta comparación, la matriz más dispersa es: Co-ocurrencia.


**¿Cómo se compara la dispersión con la matriz BoW del Laboratorio 2?**

La matriz de co-ocurrencia de este laboratorio tiene 93.51% de ceros, mientras que la BoW calculada aquí, usando las mismas 2,000 palabras, tiene 91.88%. En cambio, la BoW original del Lab 2 tuvo 99.17% de ceros, con 1,137 documentos y 23,412 términos. No es una comparación directa, porque en el Lab 2 se quitaron stopwords, se lematizó y se utilizó un vocabulario mucho mayor; al incluir palabras menos frecuentes normalmente aumenta la dispersión.

In [11]:
pares_palabras = [
    ('empresa', 'telecomunicaciones', 'Relacionadas'),
    ('gobierno', 'nacional', 'Relacionadas'),
    ('gobierno', 'federal', 'Relacionadas'),
    ('empresa', 'sánchez', 'No relacionadas'),
    ('gobierno', 'telecomunicaciones', 'No relacionadas')
]

filas_similitud = []
for palabra1, palabra2, relacion in pares_palabras:
    a, b = indice_palabras[palabra1], indice_palabras[palabra2]
    similitud = cosine_similarity(matriz_cooc[a], matriz_cooc[b])[0, 0]
    filas_similitud.append([palabra1, palabra2, relacion, similitud])

tabla_similitud = pd.DataFrame(
    filas_similitud, columns=['Palabra 1', 'Palabra 2', 'Relación', 'Coseno']
)
display(tabla_similitud.round(4))

promedio_rel = tabla_similitud.loc[tabla_similitud['Relación'] == 'Relacionadas', 'Coseno'].mean()
promedio_no_rel = tabla_similitud.loc[tabla_similitud['Relación'] == 'No relacionadas', 'Coseno'].mean()

print(f'Promedio relacionadas: {promedio_rel:.4f}')
print(f'Promedio no relacionadas: {promedio_no_rel:.4f}')


,Palabra 1,Palabra 2,Relación,Coseno
0,empresa,telecomunicaciones,Relacionadas,0.6100
1,gobierno,nacional,Relacionadas,0.7396
2,gobierno,federal,Relacionadas,0.2999
3,empresa,sánchez,No relacionadas,0.4909
4,gobierno,telecomunicaciones,No relacionadas,0.4373


Promedio relacionadas: 0.5498
Promedio no relacionadas: 0.4641


**¿La similitud coseno coincide con la intuición?**

La idea es que los pares relacionados compartan más contextos y tengan una similitud mayor. Sin embargo, el corpus habla principalmente de temas económicos y políticos, así que dos palabras poco relacionadas también pueden tener vecinos similares. La celda anterior compara los promedios de los cinco pares y muestra automáticamente si se cumple esta expectativa. En el notebook recibido esta prueba no tenía resultados porque se detenía con un error de vocabulario; por eso no se debe afirmar un valor numérico antes de ejecutarla de nuevo.

In [12]:
def top_contextos(matriz, palabra, nombre_columna, n=10):
    if palabra not in indice_palabras:
        print(f'{palabra!r} no aparece en el vocabulario de 2,000 palabras.')
        return pd.DataFrame(columns=['Palabra', 'Contexto', nombre_columna])

    idx = indice_palabras[palabra]
    valores = matriz[idx].toarray().ravel() if hasattr(matriz, 'toarray') else matriz[idx]
    orden = np.argsort(valores)[::-1]
    mejores = [i for i in orden if valores[i] > 0 and i != idx][:n]

    return pd.DataFrame({
        'Palabra': palabra,
        'Contexto': [vocabulario_2000[i] for i in mejores],
        nombre_columna: [valores[i] for i in mejores]
    })


tabla_cooc = pd.concat([
    top_contextos(matriz_cooc, palabra, 'Co-ocurrencias')
    for palabra in ['empresa', 'gobierno']
], ignore_index=True)
display(tabla_cooc)

,Palabra,Contexto,Co-ocurrencias
0,empresa,la,205
1,empresa,de,152
2,empresa,una,65
3,empresa,en,48
4,empresa,que,47
5,empresa,y,39
6,empresa,a,33
7,empresa,con,28
8,empresa,para,19
9,empresa,como,17


**¿Qué tipo de palabras dominan la lista**

Dominan palabras muy comunes, como artículos, preposiciones y conectores: para “empresa” aparecen “la” (205), “de” (152) y “una” (65), mientras que para “gobierno” destacan “el” (220), “del” (108) y “de” (84). Esto ocurre porque estas palabras aparecen cerca de muchísimos términos, aunque por sí solas no describen claramente su significado.

In [13]:
conteos = matriz_cooc.toarray().astype(np.float64)
total_cooc = conteos.sum()
if total_cooc == 0:
    raise ValueError('La matriz de co-ocurrencia no contiene pares.')

p_wc = conteos / total_cooc
p_w = p_wc.sum(axis=1, keepdims=True)
p_c = p_wc.sum(axis=0, keepdims=True)

razon = np.divide(
    p_wc, p_w * p_c,
    out=np.zeros_like(p_wc), where=(p_w * p_c) > 0
)
with np.errstate(divide='ignore'):
    matriz_ppmi = np.maximum(0, np.log2(razon))
matriz_ppmi[~np.isfinite(matriz_ppmi)] = 0

print('Forma de PPMI:', matriz_ppmi.shape)
print(f'Ceros en PPMI: {100 * np.mean(matriz_ppmi == 0):.2f}%')

Forma de PPMI: (2000, 2000)
Ceros en PPMI: 94.79%


In [14]:
tabla_ppmi = pd.concat([
    top_contextos(matriz_ppmi, palabra, 'PPMI')
    for palabra in ['empresa', 'gobierno']
], ignore_index=True)
display(tabla_ppmi.round(3))

,Palabra,Contexto,PPMI
0,empresa,telecomunicaciones,5.431
1,empresa,líder,5.155
2,empresa,llega,5.110
3,empresa,especializada,5.006
4,empresa,cualquier,4.840
5,empresa,estadounidense,4.663
6,empresa,asegurar,4.568
7,empresa,colombiana,4.535
8,empresa,criterios,4.472
9,empresa,española,4.256


**¿Qué cambió al utilizar PPMI y por qué?**

Con los conteos simples predominaban palabras como “la”, “de” y “el”, pero con PPMI aparecen asociaciones más específicas. Por ejemplo, para “empresa” destacan “telecomunicaciones” (5.431) y “líder” (5.155), y para “gobierno” aparecen “corporativo” (7.232) y “federal” (5.805). Esto pasa porque PPMI valora las co-ocurrencias que son más frecuentes de lo esperado para esas palabras, en vez de premiar solo a las palabras que más se repiten. También puede resaltar asociaciones poco frecuentes que no siempre son las más intuitivas.

In [15]:
svd = TruncatedSVD(n_components=100, random_state=42)
vectores_svd = svd.fit_transform(matriz_ppmi)

kv_svd = KeyedVectors(vector_size=100)
kv_svd.add_vectors(vocabulario_2000, vectores_svd.astype(np.float32))

ruta_svd = data / 'noticias_ppmi_svd_100.kv'
ruta_svd.parent.mkdir(parents=True, exist_ok=True)
kv_svd.save(str(ruta_svd))

print('Dimensiones de los vectores:', vectores_svd.shape)
print(f'Varianza explicada acumulada: {svd.explained_variance_ratio_.sum():.2%}')
print('Modelo guardado en:', ruta_svd)
print('Ejemplo de vector para empresa:', kv_svd['empresa'][:5]
      if 'empresa' in kv_svd else 'No disponible')

Dimensiones de los vectores: (2000, 100)
Varianza explicada acumulada: 24.94%
Modelo guardado en: data\noticias_ppmi_svd_100.kv
Ejemplo de vector para empresa: [ 8.797043  -2.7684753 -4.249758   0.523882  -3.094177 ]


**¿Por qué las filas de la matriz son vectores de palabras y qué resuelven PPMI y SVD?**

Cada fila es un vector porque muestra cuántas veces aparece una palabra cerca de las demás; dos palabras con contextos parecidos tendrán filas similares. PPMI reduce el peso de palabras muy comunes y resalta asociaciones más informativas. SVD resume esos patrones en 100 dimensiones, por lo que los vectores ocupan menos espacio y pueden captar relaciones que no dependen de compartir exactamente los mismos contextos. En este corpus se obtuvieron 2,000 vectores de 100 dimensiones, con una varianza explicada acumulada de 24.94%.

---

### 4. Entrenar Word2Vec y fastText

In [16]:
config_base = dict(vector_size=100, window=5, min_count=2, sg=1, negative=5, epochs=10, workers=1, seed=42)
configuraciones = {
    'Base (skip-gram)': {},
    'CBOW': {'sg': 0},
    'Ventana 2': {'window': 2},
    'Ventana 10': {'window': 10},
    '50 dimensiones': {'vector_size': 50},
    '300 dimensiones': {'vector_size': 300},
}
modelos_noticias = {}
tiempos = {}
for nombre, cambios in configuraciones.items():
    config = {**config_base, **cambios}
    inicio = perf_counter()
    modelos_noticias[nombre] = Word2Vec(sentences=oraciones_noticias, **config)
    tiempos[nombre] = perf_counter() - inicio
    print(f'{nombre}: {tiempos[nombre]:.2f} segundos')

Base (skip-gram): 21.99 segundos
CBOW: 6.51 segundos
Ventana 2: 11.87 segundos
Ventana 10: 33.12 segundos
50 dimensiones: 19.59 segundos
300 dimensiones: 33.91 segundos


In [17]:
preferidas = ['banco', 'gobierno', 'empresa', 'mercado', 'tecnología', 'sostenibilidad']
vocab_comun = set.intersection(*(set(m.wv.key_to_index) for m in modelos_noticias.values()))
controles = [p for p in preferidas if p in vocab_comun]
controles += [p for p, _ in frecuencias_noticias.most_common() if p in vocab_comun and p not in controles][:6-len(controles)]
assert len(controles) == 6, 'No hay seis palabras disponibles para comparar.'
print('Palabras de control:', controles)

filas_modelos = []
for nombre, modelo in modelos_noticias.items():
    fila = {'Modelo': nombre, 'sg': modelo.sg, 'window': modelo.window,
            'vector_size': modelo.vector_size, 'min_count': modelo.min_count,
            'negative': modelo.negative, 'epochs': modelo.epochs,
            'Tiempo (s)': round(tiempos[nombre], 2)}
    for palabra in controles:
        fila[palabra] = ', '.join(p for p, _ in modelo.wv.most_similar(palabra, topn=5))
    filas_modelos.append(fila)
tabla_modelos = pd.DataFrame(filas_modelos)
display(tabla_modelos)

Palabras de control: ['banco', 'gobierno', 'empresa', 'mercado', 'tecnología', 'sostenibilidad']


,Modelo,sg,window,vector_size,min_count,negative,epochs,Tiempo (s),banco,gobierno,empresa,mercado,tecnología,sostenibilidad
0,Base (skip-gram),1,5,100,2,5,10,21.99,"central, agrario, república, popular, sudameris","congreso, consejo, intervenir, mandatario, gre...","startup, emergente, innovadora, paack, plataforma","sector, colombiano, consolidándose, accionario...","estandarización, computación, definitiva, apal...","digitalización, diversidad, sociedad, biodiver..."
1,CBOW,0,5,100,2,5,10,6.51,"cordillera, supervisor, grupo, área, ranking","congreso, estado, apecic, periódico, gremio","plataforma, compañía, marca, startup, iniciativa","país, sector, mundo, turismo, territorio","digitalización, tecnológica, transformación, s...","innovación, digitalización, transformación, em..."
2,Ventana 2,1,2,100,2,5,10,11.87,"central, gnb, bce, dominicana, sudameris","congreso, bce, consejo, regulador, comité","plataforma, startup, película, matriz, veridas","sector, régimen, fútbol, conglomerado, empleo","publicidad, internet, educativa, funcionalidad...","digitalización, liderazgo, diversidad, equidad..."
3,Ventana 10,1,10,100,2,5,10,33.12,"central, agrario, república, popular, finandina","congreso, expresidente, intervenir, reforma, p...","compañía, emergente, startup, biotecnología, p...","sector, accionario, colombiano, desconocido, r...","devops, ntf, ayudarle, neurálgico, estandariza...","monográficos, prioridades, suscríbete, socieda..."
4,50 dimensiones,1,5,50,2,5,10,19.59,"central, entidad, popular, república, agrario","congreso, intervenir, gremial, anuncio, consejo","startup, compañía, innovadora, plataforma, liga","apetito, colombiano, solvencia, ganando, admin...","nube, estandarización, autentificación, portal...","diversidad, biodiversidad, salvar, sociedad, d..."
5,300 dimensiones,1,5,300,2,5,10,33.91,"central, popular, agrario, república, sudameris","congreso, intervenir, gremial, plan, mandatario","startup, emergente, compañía, biotecnología, a...","colombiano, sector, mercados, regulador, conso...","ntf, innovadora, estandarización, apalancarse,...","digitalización, alineado, diversidad, ballabri..."


**¿Las ventanas pequeñas dan vecinos más sintácticos (otras formas de la misma palabra) y las grandes, más temáticos?**

No se observa una diferencia completamente clara. Con ventana 2, “banco” se relaciona con “central”, “gnb” y “bce”, mientras que con ventana 10 aparecen “central”, “agrario” y “república”. Para “empresa”, la ventana 2 devuelve “plataforma” y “startup”, y la ventana 10 devuelve “compañía”, “emergente” y “biotecnología”. La ventana grande parece incluir asociaciones más temáticas, pero la pequeña tampoco se limita a formas gramaticales. Además, la ventana 10 tardó 34.97 segundos frente a 12.69 segundos de la ventana 2.

In [18]:
inicio = perf_counter()
modelo_quijote = Word2Vec(sentences=oraciones_quijote, **config_base)
tiempo_quijote = perf_counter() - inicio
print(f'Tiempo de entrenamiento del Quijote: {tiempo_quijote:.2f} segundos')

vocab_ambos = set(modelos_noticias['Base (skip-gram)'].wv.key_to_index) & set(modelo_quijote.wv.key_to_index)
preferidas_ambos = ['señor', 'dinero', 'casa', 'caballo', 'palabra']
palabras_ambos = [p for p in preferidas_ambos if p in vocab_ambos]
palabras_ambos += [p for p, _ in frecuencias_quijote.most_common() if p in vocab_ambos and p not in palabras_ambos][:5-len(palabras_ambos)]
assert len(palabras_ambos) == 5

comparacion_corpus = pd.DataFrame([{
    'Palabra': p,
    'Noticias': ', '.join(w for w, _ in modelos_noticias['Base (skip-gram)'].wv.most_similar(p, topn=5)),
    'Quijote': ', '.join(w for w, _ in modelo_quijote.wv.most_similar(p, topn=5))
} for p in palabras_ambos])
display(comparacion_corpus)

Tiempo de entrenamiento del Quijote: 11.82 segundos


,Palabra,Noticias,Quijote
0,señor,"tendero, arenera, fieles, unimos, comimo","álvaro, castellano, pedro, tarfe, suba"
1,dinero,"crediticio, permitía, vivir, virtuoso, recibir","dueño, haréis, capellán, perro, tema"
2,casa,"pasiva, teletrabajar, reformar, ficha, silla","aldea, compañía, presente, hacienda, casar"
3,palabra,"utópico, pendiente, empírica, técnicamente, im...","hablase, hablar, responder, respondelle, suspirar"
4,que,"identificando, sumamente, manejarlo, únicament...","glosa, brevemente, enjaulado, olalla, admira"


**¿Qué significa que el "significado viene del corpus"?**

Significa que el modelo aprende las relaciones entre palabras según los contextos en los que aparecen. Por ejemplo, en noticias “dinero” se acerca a “crediticio”, mientras que en el Quijote aparece cerca de “dueño” y “capellán”. También “casa” se relaciona con “teletrabajar” en noticias y con “aldea” y “hacienda” en el Quijote. Esto muestra que una misma palabra puede tener vecinos distintos dependiendo del tipo de texto, aunque algunos resultados también pueden deberse al tamaño del corpus y al entrenamiento.

In [19]:
inicio = perf_counter()
modelo_fasttext = FastText(sentences=oraciones_noticias, **config_base)
print(f'FastText: {perf_counter() - inicio:.2f} segundos')

candidatos_oov = {
    'Error de dedo': ['govierno', 'empreza', 'bannco'],
    'Conjugación poco común': ['gobernásemos', 'financiásemos', 'comercializásemos'],
    'Palabra inventada': ['criptoempresa', 'tecnobanco', 'mercadocripto']
}
ejemplos_oov = {}
for categoria, candidatos in candidatos_oov.items():
    palabra = next((p for p in candidatos if p not in modelo_fasttext.wv.key_to_index
                    and p not in modelos_noticias['Base (skip-gram)'].wv.key_to_index), None)
    if palabra is None:
        raise ValueError(f'No se encontró una palabra OOV para {categoria}; ajuste los candidatos.')
    ejemplos_oov[categoria] = palabra

filas_oov = []
for categoria, palabra in ejemplos_oov.items():
    vecinos = modelo_fasttext.wv.most_similar(palabra, topn=5)
    try:
        vector_w2v = modelos_noticias['Base (skip-gram)'].wv[palabra]
        resultado_w2v = f'Vector disponible: {len(vector_w2v)} dimensiones'
    except KeyError:
        resultado_w2v = 'KeyError: fuera del vocabulario'
    filas_oov.append({'Tipo': categoria, 'Palabra': palabra,
                      'Dimensiones FastText': len(modelo_fasttext.wv[palabra]),
                      'Vecinos FastText': ', '.join(w for w, _ in vecinos),
                      'Word2Vec': resultado_w2v})
display(pd.DataFrame(filas_oov))

FastText: 50.04 segundos


,Tipo,Palabra,Dimensiones FastText,Vecinos FastText,Word2Vec
0,Error de dedo,govierno,100,"invierno, infierno, gobierno, poliestireno, be...",KeyError: fuera del vocabulario
1,Conjugación poco común,gobernásemos,100,"imaginemos, sabemos, decidimos, pasemos, andemos",KeyError: fuera del vocabulario
2,Palabra inventada,criptoempresa,100,"empresa, criptografía, cripto, criptomoneda, i...",KeyError: fuera del vocabulario


**¿Por qué FastText puede representar palabras que Word2Vec no conoce?**

FastText utiliza fragmentos de caracteres (subpalabras) para construir vectores, por lo que puede representar palabras que no aparecieron durante el entrenamiento. En los resultados, “govierno” se acerca a “gobierno”, “gobernásemos” obtiene vecinos como “sabemos” y “andemos”, y “criptoempresa” se relaciona con “empresa”, “cripto” y “criptomoneda”. Las tres palabras tienen vectores de 100 dimensiones en FastText, mientras que Word2Vec devuelve un KeyError porque no están en su vocabulario. Esto no significa que FastText siempre interprete correctamente las palabras inventadas.

In [20]:
resultados = []

for nombre, modelo in modelos_noticias.items():
    similitudes = []

    for palabra in controles:
        vecinos = modelo.wv.most_similar(palabra, topn=5)
        similitudes.extend([similitud for _, similitud in vecinos])

    promedio = np.mean(similitudes)

    resultados.append({
        'Modelo': nombre,
        'Similitud promedio': promedio
    })

tabla_ranking = pd.DataFrame(resultados).sort_values(
    'Similitud promedio',
    ascending=False
).reset_index(drop=True)

display(tabla_ranking)

nombre_elegido = tabla_ranking.loc[0, 'Modelo']
modelo_elegido = modelos_noticias[nombre_elegido]

print(f'Mejor modelo según similitud promedio: {nombre_elegido}')
print(f'Similitud promedio: {tabla_ranking.loc[0, "Similitud promedio"]:.4f}')

ruta_modelo = data / 'noticias_word2vec_elegido.kv'
modelo_elegido.wv.save(str(ruta_modelo))

print('Vectores guardados en:', ruta_modelo)


,Modelo,Similitud promedio
0,CBOW,0.765923
1,50 dimensiones,0.708961
2,Ventana 2,0.636571
3,Base (skip-gram),0.611931
4,Ventana 10,0.609197
5,300 dimensiones,0.564337


Mejor modelo según similitud promedio: CBOW
Similitud promedio: 0.7659
Vectores guardados en: data\noticias_word2vec_elegido.kv
